# All-sky camera: a measured cloud record for Cerro Pachón, 2021-07 → 2026-10

> **Diagnostic** supporting [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11) (time-loss accounting); the allskycam is the primary cloud input of [weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb). Not an official metric page. Moved from rubin_notebooks; the reduction pipeline is in [bstalder/rubin_allskycam](https://github.com/bstalder/rubin_allskycam) (moved there on 2026-10-30 for handover to DM). Products are read from `ALLSKY_OUT` (default `/sdf/data/rubin/user/stalder/allsky`) and `SYSTEM_PERFORMANCE_DATA_DIR` (default `/sdf/data/rubin/user/stalder/system_performance_data`); results are written to the latter only with `ROSP11_WRITE_RESULTS=1` (`WRITE_RESULTS` in the first code cell).

> **Restructured 2026-11-06** (sp_notebooks PR #29 review). §2.2 (reduction-quality gate and LSSTCam zero points) and §3.1 (DREAM) moved here from [weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb), where they were §4.1 and §4.5. The scheduler-simulation input (old §7) moved with the downtime notebook's old §5 to [rubin_notebooks `weather_downtime_climatology.ipynb`](https://github.com/bstalder/rubin_notebooks/blob/main/earlyoperations/system_performance/weather_downtime_climatology.ipynb), as a record of the inputs forwarded to the Scheduler team in October 2026. The revision notes below are kept as history; "downtime notebook §4.1" in them is §2.2 here.


The Canon 5D Mark IV + 8–15 mm fisheye all-sky camera ("allskycam" below, to keep it distinct from LSSTCam) has taken one 30 s frame per minute every night since July 2021. This notebook analyses the reduction described in [rubin_allskycam/README.md](https://github.com/bstalder/rubin_allskycam#readme) (`allsky_reduce.py` → `regate_frames.py` → `build_completeness.py` → `cloud_metric.py` → `build_nightly.py`), whose products are read from `ALLSKY_OUT` (default: `/sdf/data/rubin/user/stalder/allsky`).

**Questions**
1. Is the allskycam's cloud index a faithful measurement of "sky unusable for Simonyi"? (Validation against the Scheduler `WEATHER` flag, Feb–Jul 2026.)
2. How does it compare with the two cloud records the weather-downtime notebook had to rely on: the meteoblue forecast nowcast and the CTIO 1975–2022 cloud database? And with DREAM?
3. What is the five-year cloud climatology at the site?
4. Side products: atmospheric transparency, zenith sky brightness, satellite/aircraft streak rates.

**The cloud index.** For every frame, every Yale-BSC star with V < 5.5, altitude > 20°, more than 25° from the moon is predicted (fisheye astrometry, ≈0.5 px rms) and checked for a detection. The *expected* number of detections is the sum of P(detect | V, alt, local sky brightness) from a clear-sky completeness model built per camera unit and calendar month from the clearest 35 % of that unit-month's frames (completeness tracks the allskycam's focus and plate scale, which both changed, and three bodies and three lenses were swapped in and out six times; see [rubin_allskycam/README.md](https://github.com/bstalder/rubin_allskycam#readme)). **`cloud_idx` = 1 − observed/expected**, clipped to [0, 1]. `frac_sky_cloudy` is the fraction of NSIDE-8 HEALPix cells with index > 0.5. The moon enters only through the model's sky-brightness axis and the 25° exclusion.

> **Revision 2026-10-04.** The downtime notebook ([weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb), §4.1) found that on 425 nights the v1 running astrometric solution sat at 1.4–2.0 px rms and left a flat 0.3–0.6 cloud-index floor on clear skies (e.g. 2025-07-17). Those nights were re-reduced with v3, and the five-year fraction of dark time with index > 0.3 dropped from 0.24 to 0.18.
>
> **Revision 2026-10-11 (v5, every night re-reduced).** Three further problems were found and fixed, and all 1,682 nights were re-reduced: (1) 1,052 nights were still on v1 and failed astrometry on half of their full-moon frames; (2) the completeness model was per calendar month, but five months contain a camera swap, and the clear-sky reference (the frames with the most stars detected) then comes mostly from the sharper unit, so the softer unit's clear nights read as partly cloudy (in February 2023 the outgoing unit's 13 nights read 0.60 cloudy against a nowcast of 0.00); (3) the star finder's shape cuts rejected the soft off-axis star images of the defocused configurations (V<5 clear-sky completeness 0.67–0.84 → 0.90–1.00). Frames are now keyed by the true night (the 2021-07 → 2022-05 archive directories are UTC dates). The five-year fraction at index > 0.3 is now **0.149** (0.127 at index > 0.5), astrometry fails on 10 % of dark frames instead of 18 %, and agreement with the independent nowcast improved (r 0.75 → 0.80). The v3 products and notebook outputs are kept in `snapshot_v3_20261010/`.
>
> **Update 2026-10-28.** The record now runs to 2026-10-27: the allskycam was down 2026-07-19 → 10-09, and the 18 nights since its restart on 2026-10-10 (body 8323 + lens 0552) are reduced with v6, which tracks the lens's zoom creep in the astrometry and keys the completeness model per service stint. The five-year fraction at index > 0.3 is **0.152** (0.129 at index > 0.5). The rise from 0.149 is October 2026, which was cloudy (8 of 18 nights mostly lost), not a change to earlier nights. Products before the catch-up are in `snapshot_pre_catchup_20261028/`.
>
> **Update 2026-10-30.** The record runs to 2026-10-29. A **deep index** (Hipparcos V < 7.5) now runs alongside the standard one: it fills about 240 of the 272 cells above 20° instead of about 105, and it measures thin cloud as an extinction in magnitudes (§6.1). The standard index and every number above are unchanged.

In [ ]:
import glob, json, os, pathlib, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
# allskycam products (bstalder/rubin_allskycam) and system-performance caches (bstalder/rubin_notebooks).
A = pathlib.Path(os.environ.get("ALLSKY_OUT", "/sdf/data/rubin/user/stalder/allsky"))
SP = pathlib.Path(
    os.environ.get(
        "SYSTEM_PERFORMANCE_DATA_DIR",
        "/sdf/data/rubin/user/stalder/system_performance_data",
    )
)
# Result files (JSON / CSV / parquet) are written to SP only when WRITE_RESULTS is True,
# so a reviewer without write access to the cache can run the notebook end to end.
WRITE_RESULTS = os.environ.get("ROSP11_WRITE_RESULTS", "0") == "1"
FLAG0, FLAG1 = 20260225, 20260713
P = dict(
    blue="#2a78d6",
    orange="#eb6834",
    aqua="#1baf7a",
    yellow="#eda100",
    magenta="#e87ba4",
    green="#008300",
    violet="#4a3aa7",
    red="#e34948",
)
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
mpl.rcParams.update(
    {
        "figure.facecolor": SURF,
        "axes.facecolor": SURF,
        "axes.edgecolor": GRID,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 1,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelcolor": INK2,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "font.size": 10,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)
MON = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
N = pd.read_parquet(A / "index/allsky_nightly.parquet")  # one row per night
CL = pd.read_parquet(A / "index/cloud_all_frames.parquet")  # one row per reduced frame
CL["t"] = pd.to_datetime(CL.mjd + 2400000.5, unit="D", origin="julian", utc=True)
print(f"{len(N)} nights, {len(CL):,} reduced frames, {N.index.min()} → {N.index.max()}")

## 1. Coverage and reduction quality

In [ ]:
qa = N.groupby("year").agg(
    nights=("n_dark", "size"),
    dark_frames=("n_dark", "sum"),
    geometry_ok=("geom_ok", "mean"),
    clear_h_per_night=("clear_h", "mean"),
    disk_R_px=("disk_R", "median"),
)
qa["lens"] = pd.cut(
    qa.disk_R_px,
    [0, 1100, 1180, 1270, 1400],
    labels=["8 mm", "9 mm (a)", "9 mm (b)", "10 mm"],
)
qa.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
m = N.groupby(N.index // 100).agg(n=("n_dark", "size"), ok=("geom_ok", "mean"))
x = pd.to_datetime(m.index.astype(str), format="%Y%m")
ax.bar(x, m.n, width=25, color=P["blue"], zorder=2, label="nights with frames")
ax.set_ylabel("nights / month")
ax.set_ylim(0, 33)
ax.set_title(
    "Archive coverage: disk 2021-07 → 2024-02, LFA 2024-02 → 2026-07 (gaps: Jul 2022, Jan 2023, Feb 2025)"
)
plt.tight_layout()

The astrometric gate "fails" on frames with too few stars to solve; those frames keep the night's running solution, so the cloud index is still defined. Geometry success is 90 % of dark frames after the v5 re-reduction (89 % in v3), and the unsolved frames are overcast. A *solved* frame can still carry a mis-registered solution (1.4–2.0 px rms) that inflates the index; after v5 no night has a median rms above 1.4 px and only 311 of 868,000 solved frames do (§2.2). 2024 is the weakest year (86 %).

## 2. Does the cloud index measure what the observers see?

The Scheduler `WEATHER` bit (set by hand by the observers since 2026-02-25) is the operational truth. We compare frame by frame, excluding planned `DOWNTIME`.

In [ ]:
st = pd.read_parquet(SP / "scheduler_observatory_status.parquet")
st = st[st.salIndex == 1].sort_index()
c = CL[(CL.day_obs >= FLAG0) & (CL.day_obs <= FLAG1)].copy()
c["status"] = st.status.reindex(c.t, method="ffill").values
c = c[c.status.notna()]
c["status"] = c.status.astype(int)
c["W"] = (c.status & 8) > 0
c["OPER"] = c.status == 2
c["D"] = (c.status & 16) > 0
b = c[~c.D & c.cloud_idx.notna() & (c.sun_alt < -12)]
rows = []
for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7]:
    pr = b.cloud_idx > thr
    rows.append(
        dict(
            threshold=thr,
            P_cloudy_given_WEATHER=pr[b.W].mean(),
            P_cloudy_given_OPERATIONAL=pr[b.OPER].mean(),
            P_cloudy_given_OPER_moon_up=pr[b.OPER & (b.moon_alt > 20)].mean(),
            P_cloudy_given_WEATHER_moon_up=pr[b.W & (b.moon_alt > 20)].mean(),
        )
    )
print(
    f"{len(b):,} dark frames with Scheduler status on {b.day_obs.nunique()} nights; WEATHER {b.W.sum():,} frames, OPERATIONAL {b.OPER.sum():,}"
)
roc = pd.DataFrame(rows).set_index("threshold").round(3)
roc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
bins = np.linspace(0, 1, 26)
axes[0].hist(
    b.cloud_idx[b.OPER],
    bins=bins,
    density=True,
    color=P["blue"],
    zorder=2,
    label="Scheduler OPERATIONAL",
)
axes[0].hist(
    b.cloud_idx[b.W],
    bins=bins,
    density=True,
    histtype="step",
    color=P["red"],
    lw=2,
    zorder=3,
    label="Scheduler WEATHER",
)
axes[0].set_xlabel("allskycam cloud index (1 − observed/expected stars)")
axes[0].set_ylabel("density")
axes[0].legend(frameon=False, fontsize=8.5)
axes[0].set_title("Frame-level separation, Feb–Jul 2026")
pn = (
    b.groupby("day_obs").agg(
        W_h=("W", "sum"), cam_h=("cloud_idx", lambda s: (s > 0.3).sum())
    )
    / 60
)
axes[1].plot([0, 12], [0, 12], color=GRID, lw=1)
axes[1].scatter(pn.cam_h, pn.W_h, s=26, color=P["blue"], edgecolor=SURF, lw=1, zorder=3)
axes[1].set_xlabel("allskycam hours with cloud index > 0.3")
axes[1].set_ylabel("Scheduler WEATHER hours")
axes[1].set_title(f"Per night, {len(pn)} nights (r = {pn.corr().iloc[0,1]:.2f})")
plt.tight_layout()
print(
    f"totals over these nights: WEATHER {pn.W_h.sum():.0f} h, allskycam cloudy (idx>0.3) {pn.cam_h.sum():.0f} h"
)

### 2.1 The disagreements

Two kinds. (a) **WEATHER set but the allskycam sees stars** (~30 % of flagged frames at threshold 0.3): the EFD limits, the dome state and the time since the last cloudy frame tell us what these are. (b) **OPERATIONAL but the allskycam sees cloud** (~7 % of operational frames): where and when.

In [ ]:
wt = pd.read_parquet(SP / "env_weather_tower_1min.parquet")
wt.index = pd.to_datetime(wt.index, utc=True)
wt["dpd"] = wt.air_temp - wt.dew_point
sh = pd.read_parquet(SP / "env_mtdome_shutter_1min.parquet")
sh.index = pd.to_datetime(sh.index, utc=True)
tm = b.t.dt.floor("1min")
m = wt.reindex(tm)
b = b.assign(
    lim=((m.rh > 80) | (m.dpd < 2) | (m.wind_speed > 15) | (m.wind_gust > 18)).values,
    open=(sh.shutter0.reindex(tm) > 95).values,
)
b = b.sort_values("t")
b["last_cloudy"] = b.t.where(b.cloud_idx > 0.3).groupby(b.day_obs).ffill()
b["min_since_cloudy"] = (b.t - b.last_cloudy).dt.total_seconds() / 60
cw = b[b.W & (b.cloud_idx <= 0.3)]
print(
    f"(a) WEATHER but allskycam-clear: {len(cw):,} frames = {len(cw)/b.W.sum():.0%} of WEATHER frames"
)
print(
    f"    outside RH/dew-point/wind limits: {cw.lim.mean():.0%} | shutter open: {cw.open.mean():.0%} | thin cloud 0.1<idx<=0.3: {(cw.cloud_idx > 0.1).mean():.0%}"
)
print(
    f"    minutes since the last allskycam-cloudy frame that night: median {cw.min_since_cloudy.median():.0f}, IQR {cw.min_since_cloudy.quantile(.25):.0f}–{cw.min_since_cloudy.quantile(.75):.0f}; never cloudy that night: {cw.min_since_cloudy.isna().mean():.0%}"
)
oc = b[b.OPER & (b.cloud_idx > 0.3)]
print(
    f"(b) OPERATIONAL but allskycam-cloudy: {len(oc):,} frames = {len(oc)/b.OPER.sum():.0%} of OPERATIONAL frames; partial sky (frac_sky_cloudy<0.5): {(oc.frac_sky_cloudy < 0.5).mean():.0%}; by month:",
    (oc.day_obs // 100).value_counts().sort_index().to_dict(),
)

**Reading.** The flagged-but-clear frames are mostly the observers *waiting*: 10 % are humidity/dew-point/wind closures, the shutter is closed in 96 % of them, 22 % show thin cloud just below the threshold (index 0.1–0.3), and the median frame is 88 minutes after the allskycam last saw cloud. The flag records the decision to stay closed, the allskycam records the sky. For a cloud *climatology* the allskycam is the right instrument; for *downtime accounting* the flag is, and the allskycam explains most of the difference between the two totals (154 h flagged vs 211 h allskycam-cloudy on the same nights: the allskycam also counts cloud during FAULT and during the first minutes before the flag is set). The operational-but-cloudy frames are concentrated in May–June 2026 and half are partial cover: the survey observing through broken cloud.

### 2.2 Is the allskycam's cloud index itself trustworthy?

Two checks before the allskycam is used as a reference. **(a) Astrometric gate.** The v1 reduction inherited a plate scale and tilt that the per-frame refinement could not always correct. On 425 such nights the running solution sat at 1.6–1.9 px rms instead of 0.6–0.9, roughly half the predicted stars fell outside the 3-px match radius, and the cloud index showed a flat 0.3–0.6 floor all night with the north/west half of the sky "cloudy" while the night report and LSSTCam said clear (e.g. 2025-07-17, 2025-07-02; un-gated five-year fraction 0.23 instead of 0.17). Those nights were re-reduced with v3 (which frees the plate scale and distortion in the final refinement) on 2026-10-04, and the whole archive again with v5 on 2026-10-11; 2025-07-17 and 2025-07-02 now have no frame above index 0.3. The gate stays in as a safeguard: the table below shows the index still rising with rms on the few frames above 1.4 px (311 of 868,000 solved frames), and nights whose median rms exceeds `CAM_RMS_MAX` are removed from the allskycam record (nights with *no* solvable frame are overcast and are kept). After v5 it removes no nights (10 after v3). The gate is applied to the frames used in this section and in §3.1; since it removes nothing, the rest of the notebook reads the frame table without it.

**(b) LSSTCam zero points.** Every on-sky science exposure with a `visit1_quicklook` zero point gives an extinction relative to the band's clear-sky zero point (90th percentile per band). This is an independent, same-sky measurement of transparency, limited to the 3.5° field being observed.

In [ ]:
# Inputs for §2.2 and §3.1 (moved from weather_downtime_analysis.ipynb, which builds the same tables):
# the gated allskycam frame record, LSSTCam science exposures and zero points, and the Scheduler flag
# on a per-minute night grid with the humidity / dew-point / wind limits.
DAY0, DAY1 = (
    20250415,
    20260713,
)  # LSSTCam window used by the downtime notebook (inclusive)
CAM_T = 0.3  # allskycam cloud index treated as "cloudy" (calibrated in the downtime notebook, §4.1)
CAM_RMS_MAX = 1.4  # nights whose running astrometric solution has median rms > this (px) are gated
LIM = dict(
    rh=80.0, dpd=2.0, wind=15.0, gust=18.0
)  # operational limits, downtime notebook §2.2

alm = pd.read_parquet(SP / "almanac_nights_2019_2027.parquet").set_index("day_obs")
ex = pd.read_parquet(SP / "consdb_lsstcam_exposures_20250415_20260720.parquet")
ex = ex[(ex.day_obs >= DAY0) & (ex.day_obs <= DAY1)]
sci = ex[(ex.img_type == "science") & (ex.can_see_sky == True)]

cam = pd.read_parquet(
    SP / "allsky_cloud_frames.parquet"
)  # written by build_nightly.py (rubin_allskycam)
cam["t"] = pd.to_datetime(cam.mjd + 2400000.5, unit="D", origin="julian", utc=True)
cam = cam.set_index("t").sort_index()
night_rms = cam[cam.ast_mode == "refine"].groupby("day_obs").ast_rms_px.median()
cam_bad_nights = night_rms.index[night_rms > CAM_RMS_MAX]
cam_all, cam = cam, cam[~cam.day_obs.isin(cam_bad_nights)]


def cam_at(times, tol="90s", c=None):
    return (cam if c is None else c).cloud_idx.reindex(
        pd.DatetimeIndex(times), method="nearest", tolerance=pd.Timedelta(tol)
    )  # nearest frame within tol, else NaN


def kappa(a, b):
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe)


def night_grid(d0, d1):
    """Per-minute grid of the dark time of nights d0..d1, with the Scheduler state bits."""
    al = alm.loc[d0:d1]
    grid = pd.date_range(
        al.t_set.min().floor("1min"),
        al.t_rise.max().ceil("1min"),
        freq="1min",
        tz="UTC",
    )
    night = np.zeros(len(grid), bool)
    dob = np.zeros(len(grid), int)
    for r in al.itertuples():
        i, j = grid.searchsorted(r.t_set), grid.searchsorted(r.t_rise)
        night[i:j] = True
        dob[i:j] = r.Index
    g = pd.DataFrame({"day_obs": dob[night]}, index=grid[night])
    g["status"] = st[["status"]].reindex(g.index, method="ffill").status
    g = g[g.status.notna()]
    g["status"] = g.status.astype(int)
    g["W"] = (g.status & 8) > 0
    g["D"] = (g.status & 16) > 0
    g["U"] = g.status == 0
    return g


g26 = night_grid(FLAG0, FLAG1).join(wt[["wind_speed", "wind_gust", "rh", "dpd"]])
g26["lim"] = (
    (g26.rh > LIM["rh"])
    | (g26.dpd < LIM["dpd"])
    | (g26.wind_speed > LIM["wind"])
    | (g26.wind_gust > LIM["gust"])
)
g26["cam"] = cam_at(g26.index).values
print(
    f"{len(sci):,} LSSTCam science exposures {DAY0}-{DAY1}; {len(cam_all):,} allskycam frames, "
    f"gate removes {len(cam_bad_nights)} nights; flag grid {len(g26) / 60:.0f} dark h on {g26.day_obs.nunique()} nights"
)

In [ ]:
# (a) index vs astrometric quality, all nights before the gate
ref = cam_all[cam_all.ast_mode == "refine"].copy()
ref["rms_bin"] = pd.cut(ref.ast_rms_px, [0, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
qa = ref.groupby("rms_bin", observed=True).agg(
    frames=("cloud_idx", "size"),
    idx_median=("cloud_idx", "median"),
    frac_idx_gt_T=("cloud_idx", lambda s: (s > CAM_T).mean()),
)
print("solved frames by rms of the astrometric solution (px):")
print(qa.round(3).T.to_string(), "\n")
nq = (
    cam_all.groupby("day_obs")
    .agg(
        q10=("cloud_idx", lambda s: s.quantile(0.1)),
        q25=("cloud_idx", lambda s: s.quantile(0.25)),
        n=("cloud_idx", "size"),
    )
    .join(night_rms.rename("rms"))
)
floor = (nq.q10 > 0.1) & (
    nq.q10 < 0.7
)  # never clear and never overcast all night: real cloud has clear gaps (q10 ≈ 0) and overcast is 1 — an intermediate floor is the artefact's signature
print(
    f"nights with an intermediate index floor (10th percentile in 0.1–0.7): {int(floor.sum())} of {len(nq)}; of these {int((floor & (nq.rms > CAM_RMS_MAX)).sum())} have rms > {CAM_RMS_MAX} px and are removed. "
    f"Gate removes {len(cam_bad_nights)} nights by year: {nq[nq.rms > CAM_RMS_MAX].groupby(nq.index[nq.rms > CAM_RMS_MAX] // 10000).size().to_dict()}; "
    f"nights with no solvable frame (kept, overcast): {int(nq.rms.isna().sum())}, median index {nq[nq.rms.isna()].q25.median():.2f}"
)
# (b) LSSTCam photometric zero points
zp = pd.read_parquet(SP / "consdb_lsstcam_zp_20250415_20260713.parquet")
refzp = zp.groupby("band").zp.quantile(0.9)
zp["ext"] = (
    refzp.reindex(zp.band).values - zp.zp
)  # mag of extinction vs the band's clear-sky zero point
zmid = pd.DatetimeIndex(zp.obs_start + (zp.obs_end - zp.obs_start) / 2)
zp["cam_pre"] = cam_at(zmid, c=cam_all).values
zp["cam"] = cam_at(zmid).values
for lab, col in [("before gate", "cam_pre"), ("after gate ", "cam")]:
    z = zp.dropna(subset=[col])
    c = z[col] > CAM_T
    print(
        f"{lab}: {len(z):,} exposures — P(allskycam > {CAM_T} | extinction < 0.1 mag) = {c[z.ext < 0.1].mean():.3f};  P(extinction > 0.3 | allskycam > {CAM_T}) = {(z.ext[c] > 0.3).mean():.2f};  "
        f"P(allskycam > {CAM_T} | extinction > 0.3) = {c[z.ext > 0.3].mean():.2f};  P(allskycam > {CAM_T} | extinction > 1 mag) = {c[z.ext > 1].mean():.2f};  r = {np.corrcoef(z[col], z.ext)[0,1]:.2f}"
    )
z = zp.dropna(subset=["cam"]).copy()
z["cb"] = pd.cut(z.cam, [-0.01, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0])
nz = z.groupby("day_obs").agg(
    n=("ext", "size"),
    cam_frac=("cam", lambda s: (s > CAM_T).mean()),
    ext_frac=("ext", lambda s: (s > 0.3).mean()),
)
nz = nz[nz.n >= 20]
print(
    f"night level, {len(nz)} observed nights with ≥20 zero points: r(allskycam-cloudy fraction, fraction of exposures with > 0.3 mag extinction) = {np.corrcoef(nz.cam_frac, nz.ext_frac)[0,1]:.2f}; "
    f"allskycam > 50 % cloudy but photometric (< 15 % of exposures extincted): {int(((nz.cam_frac > 0.5) & (nz.ext_frac < 0.15)).sum())} nights {list(nz.index[(nz.cam_frac > 0.5) & (nz.ext_frac < 0.15)])}; "
    f"allskycam < 20 % cloudy but > 50 % of exposures extincted: {int(((nz.cam_frac < 0.2) & (nz.ext_frac > 0.5)).sum())} nights"
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].bar(
    np.arange(len(qa)),
    qa.frac_idx_gt_T * 100,
    color=[P["blue"] if iv.right <= CAM_RMS_MAX else P["red"] for iv in qa.index],
    zorder=2,
)
axes[0].set_xticks(np.arange(len(qa)))
axes[0].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in qa.index], fontsize=8)
axes[0].set_xlabel("rms of the frame's astrometric solution (px)")
axes[0].set_ylabel(f"% of solved frames with index > {CAM_T}")
axes[0].set_title("Index inflates with poor astrometry (red = gated)", fontsize=10)
tab = z.groupby("cb", observed=True).ext.quantile([0.1, 0.5, 0.9]).unstack()
x = np.arange(len(tab))
axes[1].fill_between(
    x,
    tab[0.1],
    tab[0.9],
    color=P["blue"],
    alpha=0.15,
    lw=0,
    label="10–90 % of exposures",
)
axes[1].plot(x, tab[0.5], color=P["blue"], marker="o", ms=5, label="median")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in tab.index], fontsize=8)
axes[1].set_xlabel("allskycam cloud index at mid-exposure")
axes[1].set_ylabel("LSSTCam extinction (mag)")
axes[1].set_title(
    f"LSSTCam extinction vs allskycam index ({len(z):,} exposures)", fontsize=10
)
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** The gate is a reduction-quality cut, not a weather cut, and after the v5 re-reduction it removes nothing. An allskycam index above the threshold occurs on 0.3 % of photometric exposures (1.3 % in v3), and 90 % of allskycam-cloudy exposures carry more than 0.3 mag of extinction (three quarters in v3). The reverse is weaker: the allskycam flags 36 % of the exposures with more than 0.3 mag of extinction, because thin cirrus dims stars without hiding them and because a 3.5° field can be clouded while the all-sky mean is not; even above 1 mag the allskycam catches only about two thirds. The allskycam is therefore a measure of *sky unusable for star counting*, i.e. the operator's "can the AOS find stars" criterion, and a lower bound on the time with non-photometric transparency.

## 3. Allskycam vs the meteoblue nowcast and vs DREAM

The weather-downtime notebook had no sky measurement and used the meteoblue forecast valid for each hour (threshold 70 % total cloud cover, tuned to the flag) as a cloud proxy. The allskycam overlaps it on ~1000 nights.

In [ ]:
o = N[N.nowcast_cloud70_frac.notna()]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[0].scatter(
    o.nowcast_cloud70_frac,
    o.cloudy30,
    s=10,
    alpha=0.5,
    color=P["blue"],
    edgecolor="none",
    zorder=3,
)
axes[0].set_xlabel("meteoblue nowcast: fraction of night with cloud cover > 70 %")
axes[0].set_ylabel("allskycam: fraction of night with cloud index > 0.3")
axes[0].set_title(
    f"Nightly, {len(o)} nights 2022-12 → 2026-07 (r = {o[['cloudy30','nowcast_cloud70_frac']].corr().iloc[0,1]:.2f})"
)
miss = ((o.nowcast_cloud70_frac < 0.1) & (o.cloudy30 > 0.5)).sum()
false = ((o.nowcast_cloud70_frac > 0.5) & (o.cloudy30 < 0.1)).sum()
axes[0].text(
    0.02,
    0.93,
    f"nowcast clear, allskycam cloudy: {miss} nights\nnowcast cloudy, allskycam clear: {false} nights",
    transform=axes[0].transAxes,
    fontsize=8.5,
    color=INK2,
    va="top",
)
d = N[N.dream_n.notna() & (N.dream_n > 10)]
axes[1].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[1].scatter(
    d.dream_cloudy_frac,
    d.cloudy30,
    s=14,
    alpha=0.7,
    color=P["orange"],
    edgecolor="none",
    zorder=3,
)
axes[1].set_xlabel(
    "DREAM: fraction of night with >50 % of sky (alt>30°) at extinction > 0.1 mag"
)
axes[1].set_ylabel("allskycam: fraction with cloud index > 0.3")
axes[1].set_title(
    f"Allskycam vs DREAM, {len(d)} nights 2025-06 → 2026-06 (r = {d[['cloudy30','dream_cloudy_frac']].corr().iloc[0,1]:.2f})",
    fontsize=10,
)
plt.tight_layout()
print(
    f"same-night means: allskycam {o.cloudy30.mean():.3f}, nowcast {o.nowcast_cloud70_frac.mean():.3f}  → the nowcast under-reports cloud by a factor {o.cloudy30.mean()/o.nowcast_cloud70_frac.mean():.1f}"
)
print(
    f"DREAM nights: allskycam {d.cloudy30.mean():.3f} vs DREAM {d.dream_cloudy_frac.mean():.3f}"
)

**Reading.** The nowcast is one-sided: when it says cloudy the allskycam almost always agrees (8 exceptions in ~1000 nights), but it misses about one night in forty that the allskycam shows as mostly cloudy (25 nights), and it under-reports the cloudy fraction by a factor 1.2 (1.3 in v3, 1.9 before the v3 re-reduction; both earlier factors were partly reduction artefacts). This is exactly the failure mode you would expect from calibrating a forecast threshold to the operators' flag during one season. A climatology that rests on the nowcast is therefore biased low. DREAM (305 nights) agrees with the allskycam on the clear/cloudy split at the night level (r ≈ 0.8); at DREAM's 0.1 mag threshold, which sits on its zero-point floor, it reports more cloudy time (0.19 vs 0.11). §3.1 compares the two frame by frame and cell by cell at a 0.5 mag threshold, where they agree.

### 3.1 A second cloud instrument: DREAM, frame by frame

DREAM (the Dutch Rubin Enhanced Atmospheric Monitor) is a five-camera photometric cloud monitor at the summit. Its `cloud_sys` product is an NSIDE-32 HEALPix map of **extinction in magnitudes** (per-pixel zero-point deficit of the stars it photometers against a clear-sky reference, stacked over the last five epochs ≈ 2–3 min), published to the LFA every ≈ 32 s from **2025-06-25 to 2026-06-21** (315 nights, 349 k maps; it stopped before the storm). It measures the quantity the allskycam does not: how much the stars are *dimmed*, not whether they are *lost*. `build_dream_dense.py` (rubin_allskycam) pulls every map, rotates it to horizon coordinates, and writes three caches: whole-sky scalars per map (`dream_maps.parquet`), the DREAM extinction in a 1.75° disc around every LSSTCam science pointing (`dream_lsstcam_exposures.parquet`), and, for every allskycam frame, the whole-sky DREAM metrics plus a cell-by-cell comparison on the allskycam's NSIDE-8 grid (`dream_allskycam_cells.parquet`). Four questions: (a) do the two sky monitors see the same clouds, frame by frame and cell by cell; (b) which one agrees better with LSSTCam's own zero points; (c) how does DREAM compare with the Scheduler flag; (d) does adding DREAM change the nightly weather-loss fraction.

Where the two disagree the sign is diagnostic. The allskycam is a star-*count* metric and the DREAM scalar used here is the *median extinction above 30° altitude*, so DREAM-only cloud should be thin or uniform cloud (stars dimmed but still detected) and allskycam-only cloud should be the moon floor of §2.2 or stars lost without much extinction. Both are checked below against LSSTCam and against whether the survey was actually observing.


In [ ]:
# ---- DREAM: dense cloud_sys record (every map, ~32 s), built by build_dream_dense.py (rubin_allskycam) from the LFA ------------------------------------------
dm = pd.read_parquet(SP / "dream_maps.parquet")
dm["t"] = pd.to_datetime(dm.mjd + 2400000.5, unit="D", origin="julian", utc=True)
dm = dm.set_index("t").sort_index()
de = pd.read_parquet(
    SP / "dream_lsstcam_exposures.parquet"
)  # LSSTCam science exposures: median DREAM extinction in a 1.75° disc around the pointing, nearest map within 90 s
dc = pd.read_parquet(
    SP / "dream_allskycam_cells.parquet"
)  # allskycam frames: nearest map within 60 s, whole-sky DREAM metrics and the NSIDE-8 cell-by-cell comparison
dc = dc[~dc.day_obs.isin(cam_bad_nights)].copy()
DREAM_T = 0.5  # DREAM "cloudy": median extinction above 30° altitude > 0.5 mag (κ-optimal against both the allskycam and the flag, see below)
dc["cam_c"] = dc.cloud_idx > CAM_T
dc["dr_c"] = dc.dream_med_ext_alt30 > DREAM_T
dc["moon_up"] = dc.moon_alt > 0
cad = (
    dm.groupby("day_obs")
    .apply(lambda d: d.index.to_series().diff().median().total_seconds())
    .median()
)
print(
    f"DREAM: {len(dm):,} cloud maps on {dm.day_obs.nunique()} nights {dm.day_obs.min()} → {dm.day_obs.max()}, median cadence {cad:.0f} s; "
    f"{len(dc):,} allskycam frames with a DREAM map within 60 s ({dc.day_obs.nunique()} nights); {len(de):,} LSSTCam science exposures with a DREAM map within 90 s"
)
# (a) frame level: what does DREAM report as a function of the allskycam index?
dc["cb"] = pd.cut(dc.cloud_idx, [-0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0])
fa = dc.groupby("cb", observed=True).agg(
    frames=("cloud_idx", "size"),
    dream_med_ext=("dream_med_ext_alt30", "median"),
    q10=("dream_med_ext_alt30", lambda s: s.quantile(0.1)),
    q90=("dream_med_ext_alt30", lambda s: s.quantile(0.9)),
    dream_frac_sky_gt_0p3=("dream_frac_alt30_ext_gt_0p3", "median"),
    P_dream_cloudy=("dr_c", "mean"),
    r_cells=("r_cells", "median"),
)
fa["cell_agreement"] = dc.groupby("cb", observed=True).apply(
    lambda d: (d.n_cc + d.n_00).sum() / (d.n_cc + d.n_00 + d.n_c0 + d.n_0c).sum()
)
print(
    "\nDREAM by allskycam cloud-index bin (cell agreement: NSIDE-8 cells, allskycam index > 0.5 vs DREAM mean extinction > 0.3 mag):"
)
print(fa.round(3).T.to_string())
print(
    f"\nframe-level correlation of the allskycam index with DREAM median extinction r = {np.corrcoef(dc.cloud_idx, dc.dream_med_ext_alt30.fillna(0))[0,1]:.2f}, with the DREAM fraction of sky above 0.3 mag r = {np.corrcoef(dc.cloud_idx, dc.dream_frac_alt30_ext_gt_0p3.fillna(0))[0,1]:.2f}"
)
part = dc[(dc.cloud_idx > 0.2) & (dc.cloud_idx < 0.8)]
tot = (part.n_cc + part.n_00 + part.n_c0 + part.n_0c).sum()
print(
    f"partial-cloud frames (0.2 < index < 0.8, {len(part):,}): median cell-by-cell correlation {part.r_cells.median():.2f}; cells agree {(part.n_cc + part.n_00).sum()/tot:.0%}; "
    f"of allskycam-cloudy cells DREAM calls {part.n_c0.sum()/(part.n_cc + part.n_c0).sum():.0%} clear; of DREAM-cloudy cells the allskycam calls {part.n_0c.sum()/(part.n_0c + part.n_cc).sum():.0%} clear (0.3 mag is a lower bar than 'half the stars gone')"
)
# (b) the disagreements: moon, whether LSSTCam was observing, and what LSSTCam measured
cls = np.select(
    [dc.cam_c & dc.dr_c, dc.cam_c & ~dc.dr_c, ~dc.cam_c & dc.dr_c],
    ["both cloudy", "allskycam only", "DREAM only"],
    "both clear",
)
ex_mid = (
    pd.DatetimeIndex(sci.obs_start + (sci.obs_end - sci.obs_start) / 2)
    .tz_convert("UTC")
    .sort_values()
)
ft = pd.DatetimeIndex(
    pd.to_datetime(dc.mjd + 2400000.5, unit="D", origin="julian", utc=True)
)
pos = ex_mid.searchsorted(ft)
near = np.full(len(ft), np.inf)
for k in (pos - 1, pos):
    kk = np.clip(k, 0, len(ex_mid) - 1)
    near = np.minimum(near, np.abs((ex_mid[kk] - ft).total_seconds()))
dc["observing"] = (
    near <= 120
)  # an LSSTCam science exposure mid-point within 2 min of the allskycam frame
zx = zp.dropna(subset=["cam"]).copy()
zx["t"] = pd.to_datetime(zx.obs_start + (zx.obs_end - zx.obs_start) / 2, utc=True)
mm = pd.merge_asof(
    zx[["t", "ext"]].sort_values("t"),
    dc.assign(t=ft)[["t", "cam_c", "dr_c"]].sort_values("t"),
    on="t",
    direction="nearest",
    tolerance=pd.Timedelta("90s"),
).dropna()
cls2 = np.select(
    [mm.cam_c & mm.dr_c, mm.cam_c & ~mm.dr_c, ~mm.cam_c & mm.dr_c],
    ["both cloudy", "allskycam only", "DREAM only"],
    "both clear",
)
dis = dc.groupby(cls).agg(
    hours=("cam_c", lambda s: len(s) / 60),
    moon_up=("moon_up", "mean"),
    cam_idx_median=("cloud_idx", "median"),
    dream_ext_median=("dream_med_ext_alt30", "median"),
    P_LSSTCam_observing=("observing", "mean"),
)
dis = dis.join(
    mm.groupby(cls2).ext.agg(
        lsstcam_exposures="size",
        lsstcam_ext_median="median",
        P_lsstcam_ext_gt_0p3=lambda s: (s > 0.3).mean(),
    )
)
print(
    f"\nallskycam (index > {CAM_T}) × DREAM (median extinction > {DREAM_T} mag) classification of the allskycam minutes:"
)
print(dis.round(3).to_string())

In [ ]:
# (c) three-way check with LSSTCam zero points: DREAM extinction in the 1.75° disc around the pointing vs the exposure's own extinction
j = (
    de[de.n_pix >= 4]
    .merge(zp[["exposure_id", "ext", "cam"]], on="exposure_id")
    .dropna(subset=["cam"])
)
j = j[~j.day_obs.isin(cam_bad_nights)].copy()
j["moon_up"] = (
    cam.moon_alt.reindex(
        pd.DatetimeIndex(
            pd.to_datetime(j.dream_mjd + 2400000.5, unit="D", origin="julian", utc=True)
        ),
        method="nearest",
        tolerance=pd.Timedelta("90s"),
    ).values
    > 0
)
ph = j[j.ext < 0.1]
print(
    f"{len(j):,} LSSTCam science exposures ({j.day_obs.nunique()} nights) with DREAM, allskycam and a zero point. On photometric exposures (LSSTCam extinction < 0.1 mag) DREAM reports {ph.ext_dream.median():.3f} mag median, robust σ {1.4826*(ph.ext_dream-ph.ext_dream.median()).abs().median():.3f} mag, "
    f"P(DREAM > 0.1) = {(ph.ext_dream > 0.1).mean():.2f}, P(DREAM > 0.3) = {(ph.ext_dream > 0.3).mean():.4f};  P(allskycam > {CAM_T} | photometric) = {(ph.cam > CAM_T).mean():.3f}"
)
s = j[(j.ext > 0.3) & (j.ext < 3)]
print(
    f"DREAM vs LSSTCam extinction: r = {j[j.ext < 5][['ext_dream', 'ext']].corr().iloc[0,1]:.2f}; median DREAM / LSSTCam ratio {(s.ext_dream/s.ext).median():.2f} for 0.3 < ext < 3 mag (moon down {(s.ext_dream/s.ext)[~s.moon_up].median():.2f}, moon up {(s.ext_dream/s.ext)[s.moon_up].median():.2f})"
)
rows = []
for lo, hi in [
    (-9, 0.05),
    (0.05, 0.1),
    (0.1, 0.2),
    (0.2, 0.3),
    (0.3, 0.5),
    (0.5, 1),
    (1, 2),
    (2, 9),
]:
    s = j[(j.ext > lo) & (j.ext <= hi)]
    rows.append(
        dict(
            lsstcam_ext=f"{max(lo,0):g}–{min(hi,5):g}",
            exposures=len(s),
            dream_median=s.ext_dream.median(),
            dream_q10=s.ext_dream.quantile(0.1),
            dream_q90=s.ext_dream.quantile(0.9),
            P_dream_gt_0p1=(s.ext_dream > 0.1).mean(),
            P_dream_gt_0p3=(s.ext_dream > 0.3).mean(),
            P_allskycam_gt_T=(s.cam > CAM_T).mean(),
        )
    )
tw = pd.DataFrame(rows).set_index("lsstcam_ext")
print("\nDREAM field extinction and allskycam index by LSSTCam extinction bin (mag):")
print(tw.round(3).T.to_string())
for mu, lab in [(False, "moon down"), (True, "moon up  ")]:
    s = j[j.moon_up == mu]
    e3 = s.ext > 0.3
    print(
        f"{lab}: {len(s):,} exposures, P(LSSTCam ext > 0.3) = {e3.mean():.3f};  P(DREAM field > 0.3 | ext > 0.3) = {(s.ext_dream[e3] > .3).mean():.2f} vs P(allskycam > {CAM_T} | ext > 0.3) = {(s.cam[e3] > CAM_T).mean():.2f};  "
        f"P(DREAM > 0.3 | ext > 1) = {(s.ext_dream[s.ext > 1] > .3).mean():.2f} vs P(allskycam > {CAM_T} | ext > 1) = {(s.cam[s.ext > 1] > CAM_T).mean():.2f};  P(allskycam > {CAM_T} | photometric) = {(s.cam[s.ext < .1] > CAM_T).mean():.3f}, P(DREAM > 0.3 | photometric) = {(s.ext_dream[s.ext < .1] > .3).mean():.4f}"
    )
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
x = np.arange(len(fa))
axes[0].fill_between(
    x, fa.q10, fa.q90, color=P["orange"], alpha=0.15, lw=0, label="10–90 % of frames"
)
axes[0].plot(x, fa.dream_med_ext, color=P["orange"], marker="o", ms=5, label="median")
axes[0].axhline(DREAM_T, color=GRID, lw=1)
axes[0].text(
    0.05, DREAM_T + 0.1, f"DREAM cloudy threshold {DREAM_T} mag", color=INK2, fontsize=8
)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{iv.left:g}–{iv.right:g}" for iv in fa.index], fontsize=8)
axes[0].set_xlabel("allskycam cloud index")
axes[0].set_ylabel("DREAM median extinction, alt > 30° (mag)")
axes[0].set_title(f"DREAM vs allskycam, {len(dc):,} frames", fontsize=10)
axes[0].legend(frameon=False, fontsize=8.5, loc="upper left")
x = np.arange(len(tw))
mids = np.array([0.025, 0.075, 0.15, 0.25, 0.4, 0.75, 1.5, 3.0])
axes[1].fill_between(
    x,
    tw.dream_q10,
    tw.dream_q90,
    color=P["orange"],
    alpha=0.15,
    lw=0,
    label="10–90 % of exposures",
)
axes[1].plot(
    x,
    tw.dream_median,
    color=P["orange"],
    marker="o",
    ms=5,
    label="DREAM at the pointing (median)",
)
axes[1].plot(x, mids, color=GRID, lw=1, ls="--", label="1:1")
axes[1].plot(x, 0.76 * mids, color=INK2, lw=1, ls=":", label="0.76 × LSSTCam")
axes[1].set_xticks(x)
axes[1].set_xticklabels(tw.index, fontsize=8)
axes[1].set_xlabel("LSSTCam extinction (mag, vs the band's clear-sky zero point)")
axes[1].set_ylabel("DREAM extinction at the pointing (mag)")
axes[1].set_title(f"DREAM vs LSSTCam, {len(j):,} exposures", fontsize=10)
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** The two monitors see the same sky. Frame by frame the allskycam index and DREAM's median extinction correlate at r = 0.90, and on partial-cloud frames the cell-by-cell pattern matches at r ≈ 0.67 (the side-by-side frames from `render_cam_dream.py` (rubin_allskycam) under `$ALLSKY_OUT/demo/` show the same gaps in both). The disagreements have the expected signs and LSSTCam adjudicates both:

* **DREAM-only cloud (68 h, 2 % of the overlap)** is thin or uniform cloud: the allskycam index is 0.17 (stars dimmed, still detected) while DREAM reports 0.8 mag, LSSTCam was observing through it half the time, and 94 % of those exposures carry > 0.3 mag of extinction (median 0.93 mag). This is the ~60 % of extincted exposures the allskycam misses in §2.2, now measured directly. Only 10 % of these minutes are moonlit.
* **Allskycam-only cloud (45 h, 58 h in v3)** is 94 % moonlit: the bright-moon completeness floor of §2.2. DREAM sees 0.12 mag, LSSTCam sees 0.33 mag median, and 54 % of the exposures do exceed 0.3 mag, so it is about half real thin cloud under the moon and half the artefact.
* On **photometric** LSSTCam exposures DREAM reports 0.04 ± 0.06 mag with no false positives above 0.3 mag (5 in 10 000), and the allskycam at index 0.3 does as well (0.1 %, 0.5 % in v3). DREAM is the better *transparency* monitor: for exposures LSSTCam measures above 0.3 mag of extinction it fires 79 % of the time with the moon down (allskycam 38 %), and above 1 mag 95 % (allskycam 65 %); with the moon up the two converge (58 vs 49 %) because DREAM's own photometric zero point degrades under moonlight. DREAM under-reads LSSTCam by a constant factor **≈ 0.75** (median ratio 0.74 over 0.3–3 mag; 0.78 moon down, 0.64 moon up): a 1.75° disc of an NSIDE-32 map smooths the cloud that the 3.5° LSSTCam field averages differently, and DREAM's reference zero point is itself set on slightly non-photometric nights.

So the allskycam measures *unobservable* sky (stars gone) and DREAM measures *non-photometric* sky (stars dimmed); at a 0.5 mag threshold they describe the same cloudy hours, and the 0.1 mag DREAM threshold reaches into its zero-point floor (10 % of photometric exposures, and a January floor of 18 % of dark time with the allskycam at 0) and should not be used for downtime.


In [ ]:
# (d) against the Scheduler WEATHER flag (2026-02-25 → 06-21, where DREAM runs), on the same minutes as the allskycam
dm1 = dm[["frac_alt30_ext_gt_0p3", "med_ext_alt30"]].resample("1min").mean()
g26["dream_ext"] = dm1.med_ext_alt30.reindex(
    g26.index, method="nearest", tolerance=pd.Timedelta("90s")
)
g26["dream_f03"] = dm1.frac_alt30_ext_gt_0p3.reindex(
    g26.index, method="nearest", tolerance=pd.Timedelta("90s")
)
bd = g26[~g26.D & ~g26.U].dropna(subset=["dream_ext", "cam"])
print(
    f"{len(bd)/60:.0f} dark hours with Scheduler status, allskycam and DREAM (DREAM covers {g26.dream_ext.notna().mean():.0%} of the flag-era minutes; it stopped on 2026-06-21): P(WEATHER) = {bd.W.mean():.3f}"
)
rows = []
for lab, pred in [
    (f"allskycam index > {CAM_T}", bd.cam > CAM_T),
    ("allskycam index > 0.5", bd.cam > 0.5),
    ("allskycam index > 0.6", bd.cam > 0.6),
    ("DREAM median ext > 0.1 mag", bd.dream_ext > 0.1),
    ("DREAM median ext > 0.3 mag", bd.dream_ext > 0.3),
    (f"DREAM median ext > {DREAM_T} mag", bd.dream_ext > DREAM_T),
    ("DREAM median ext > 1 mag", bd.dream_ext > 1.0),
    ("DREAM: > 50 % of sky above 0.3 mag", bd.dream_f03 > 0.5),
    (
        f"allskycam > {CAM_T} OR DREAM > {DREAM_T}",
        (bd.cam > CAM_T) | (bd.dream_ext > DREAM_T),
    ),
    (
        f"allskycam > {CAM_T} AND DREAM > {DREAM_T}",
        (bd.cam > CAM_T) & (bd.dream_ext > DREAM_T),
    ),
]:
    rows.append(
        dict(
            proxy=lab,
            hours=pred.sum() / 60,
            WEATHER_h=bd.W.sum() / 60,
            recall=pred[bd.W].mean(),
            precision=bd.W[pred].mean(),
            P_proxy_given_OPERATIONAL=pred[bd.status == 2].mean(),
            kappa=kappa(bd.W.values, pred.values),
        )
    )
fl = pd.DataFrame(rows).set_index("proxy")
print(fl.round(3).to_string())
cc, dd = bd.cam > CAM_T, bd.dream_ext > DREAM_T
print(
    f"\nP(WEATHER | both cloudy) = {bd.W[cc & dd].mean():.2f} ({(cc & dd).sum()/60:.0f} h);  P(WEATHER | allskycam only) = {bd.W[cc & ~dd].mean():.2f} ({(cc & ~dd).sum()/60:.0f} h);  "
    f"P(WEATHER | DREAM only) = {bd.W[~cc & dd].mean():.2f} ({(~cc & dd).sum()/60:.0f} h);  P(WEATHER | neither) = {bd.W[~cc & ~dd].mean():.3f}"
)
print(
    f"DREAM median extinction while WEATHER (inside limits) — quartiles: {bd.dream_ext[bd.W & ~bd.lim].quantile([.25,.5,.75]).round(2).tolist()};  while OPERATIONAL: {bd.dream_ext[bd.status == 2].quantile([.25,.5,.75]).round(3).tolist()}"
)

In [ ]:
# (e) nightly: fraction of dark time cloudy by each instrument, on the nights both cover ≥ 80 % of the dark minutes
dc["union"] = dc.cam_c | dc.dr_c
dc["inter"] = dc.cam_c & dc.dr_c
dc["dr_thin"] = dc.dream_med_ext_alt30 > 0.1
dn = dc.groupby("day_obs").agg(
    n=("cam_c", "size"),
    cam_frac_d=("cam_c", "mean"),
    cam_frac50_d=("cloud_idx", lambda s: (s > 0.5).mean()),
    dream_frac=("dr_c", "mean"),
    dream_frac_thin=("dr_thin", "mean"),
    union_frac=("union", "mean"),
    inter_frac=("inter", "mean"),
)
dn["dark_h"] = alm.dark_hours_12.reindex(dn.index)
dn["cov"] = dn.n / 60 / dn.dark_h
dv = dn[dn["cov"] > 0.8].copy()
dv["month"] = dv.index.astype(str).str[4:6].astype(int)
eqm = (
    dv.groupby("month")[
        [
            "cam_frac_d",
            "cam_frac50_d",
            "dream_frac",
            "dream_frac_thin",
            "union_frac",
            "inter_frac",
        ]
    ]
    .mean()
    .reindex(range(1, 13))
)
eq = eqm.mean()
print(
    f"{len(dv)} nights ({dv.index.min()} → {dv.index.max()}) where DREAM and the allskycam both cover ≥ 80 % of the dark minutes. Fraction of dark time cloudy, equal-month mean:\n"
    f"   allskycam > {CAM_T}: {eq.cam_frac_d:.3f} | allskycam > 0.5: {eq.cam_frac50_d:.3f} | DREAM > {DREAM_T} mag: {eq.dream_frac:.3f} | DREAM > 0.1 mag (thin, at DREAM's zero-point floor): {eq.dream_frac_thin:.3f} | either: {eq.union_frac:.3f} | both: {eq.inter_frac:.3f}"
)
print(
    f"night by night r(allskycam, DREAM) = {np.corrcoef(dv.cam_frac_d, dv.dream_frac)[0,1]:.2f}; nights fully lost (> 0.9): allskycam {(dv.cam_frac_d > .9).mean():.3f}, DREAM {(dv.dream_frac > .9).mean():.3f}, either {(dv.union_frac > .9).mean():.3f}; "
    f"clear (< 0.1): {(dv.cam_frac_d < .1).mean():.2f} / {(dv.dream_frac < .1).mean():.2f} / {(dv.union_frac < .1).mean():.2f}"
)
big = dv[(dv.cam_frac_d - dv.dream_frac).abs() > 0.3]
print(
    f"nights where the two differ by > 0.3 of the night: {len(big)} — DREAM higher: {list(big.index[big.dream_frac > big.cam_frac_d])}; allskycam higher: {list(big.index[big.cam_frac_d > big.dream_frac])}"
)
print(eqm.round(3).T.to_string())
dream_res = dict(
    nights=int(len(dv)),
    span=[int(dv.index.min()), int(dv.index.max())],
    dream_threshold_mag=DREAM_T,
    equal_month_mean=eq.round(4).to_dict(),
    monthly=eqm.round(4).to_dict(),
    flag_calibration=fl.round(4).to_dict(orient="index"),
    lsstcam_bins=tw.round(4).to_dict(orient="index"),
    classes=dis.round(4).to_dict(orient="index"),
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
mo = np.arange(1, 13)
axes[0].plot(
    mo,
    eqm.cam_frac_d * 100,
    color=P["blue"],
    marker="o",
    ms=4,
    label=f"allskycam index > {CAM_T}",
)
axes[0].plot(
    mo,
    eqm.dream_frac * 100,
    color=P["orange"],
    marker="o",
    ms=4,
    label=f"DREAM median ext > {DREAM_T} mag",
)
axes[0].plot(mo, eqm.union_frac * 100, color=INK2, lw=1.2, ls="--", label="either")
axes[0].plot(
    mo,
    eqm.dream_frac_thin * 100,
    color=P["orange"],
    lw=1.2,
    ls=":",
    label="DREAM > 0.1 mag (thin cloud)",
)
axes[0].set_xticks(mo)
axes[0].set_xticklabels(list("JFMAMJJASOND"))
axes[0].set_ylabel("% of dark time cloudy")
axes[0].set_title(
    f"Monthly, {len(dv)} nights with both instruments (Jun 2025 → Jun 2026)",
    fontsize=10,
)
axes[0].legend(frameon=False, fontsize=8.5)
axes[1].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[1].scatter(
    dv.cam_frac_d,
    dv.dream_frac,
    s=22,
    color=P["blue"],
    edgecolor=SURF,
    linewidth=0.8,
    zorder=3,
)
for d, r in big.iterrows():
    axes[1].annotate(
        str(d),
        (r.cam_frac_d, r.dream_frac),
        textcoords="offset points",
        xytext=(4, -3),
        fontsize=6.5,
        color=INK2,
    )
axes[1].set_xlabel(f"allskycam: fraction of dark time with index > {CAM_T}")
axes[1].set_ylabel(f"DREAM: fraction with median ext > {DREAM_T} mag")
axes[1].set_title(
    f"Night by night, r = {np.corrcoef(dv.cam_frac_d, dv.dream_frac)[0,1]:.2f}",
    fontsize=10,
)
plt.tight_layout()

**Reading.** Against the Scheduler flag DREAM behaves like a less specific allskycam: at any threshold its recall is comparable (0.52–0.70) but its precision is lower (0.31–0.43 vs 0.48–0.57) and κ peaks at 0.40 (1 mag) against 0.52 for the allskycam at 0.5. The reason is in the two-way split: hours both instruments call cloudy are flagged 46 % of the time, allskycam-only hours 56 %, but DREAM-only hours just 7 %, the thin cloud through which the survey keeps observing (above). The operators close for *lost stars*, not for extinction, which is also what [weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb) (§4.1) found from the κ peak at index 0.5. The union of the two does not help (κ 0.45) because it adds exactly the thin-cloud hours the operators ignore; the intersection is the most specific proxy of all (precision 0.46) at the cost of recall.

At the night level the two agree at r = 0.91 over 283 nights, with the same bimodality (80 % of nights clear by either, 2–3 % fully lost). Over Jun 2025 → Jun 2026 DREAM at 0.5 mag gives **0.106** of dark time cloudy against the allskycam's 0.098 (0.075 at index 0.5); either instrument 0.120, both 0.083. The eight nights that differ by more than 0.3 split cleanly: DREAM higher on five thin-cloud nights (2025-09-12, 2025-11-17, 2026-05-19, -05-22, -06-12: where LSSTCam observed, 54–100 % of exposures had > 0.3 mag of extinction while the allskycam index stayed low) and the allskycam higher on three full-moon nights (2026-03-04, -05-27, -05-29; moon altitude up to 64–84°, DREAM 0.0–0.1 mag), the moon floor again. **Conclusion for §5:** adding DREAM does not change the headline. Its one year overlaps the clear 2025–26 season, it adds ≈ +0.02 of dark time as thin cloud when taken as a union, and it confirms the allskycam's bright-moon floor as the dominant known bias. DREAM's own value is as the *transparency* record: the fraction of observed exposures that are non-photometric, which the §5 "unobservable time" numbers deliberately exclude. `dream_nightly.parquet` carries the per-night fractions and `allsky_cloud_results.json` a `dream` block.


## 4. Night classes from the downtime notebook, seen by the allskycam

In [ ]:
cc = (
    N[N["class"].notna()]
    .groupby("class")
    .agg(
        nights=("cloudy30", "size"),
        camera_cloudy_frac_median=("cloudy30", "median"),
        camera_cloudy_frac_mean=("cloudy30", "mean"),
    )
)
cc.round(2)

Weather-closed nights identified by the flag or the EFD proxy are 87–91 % cloudy (median) by the allskycam; hardware/engineering closures and observed nights have a median of 0 and a mean of 6–8 %. The "closed: weather (log text)" class is the exception: those are the humidity / dew-point closures of Dec 2025–Jan 2026 under a clear sky, correctly not seen as cloud. The nine "undetermined" nights were cloudy: the allskycam resolves them.

## 5. Five-year cloud climatology

In [ ]:
clim = N.groupby("month").agg(
    cam20=("cloudy20", "mean"),
    cam30=("cloudy30", "mean"),
    cam50=("cloudy50", "mean"),
    idx_mean=("idx_mean", "mean"),
    nights=("cloudy30", "size"),
)
# Monthly fractions of the other records, written by rubin_notebooks weather_downtime_climatology.ipynb.
prev = pd.read_csv(SP / "weather_downtime_monthly_climatology.csv").set_index("month")
clim["nowcast_proxy_2023_26"] = prev.iloc[:, 1]
clim["CTIO_1975_2022"] = prev.iloc[:, 3]
clim["Scheduler_WEATHER_2026"] = prev.iloc[:, 0]
fig, ax = plt.subplots(figsize=(10, 4))
mo = np.arange(1, 13)
ax.fill_between(
    mo,
    clim.cam50 * 100,
    clim.cam20 * 100,
    color=P["blue"],
    alpha=0.15,
    lw=0,
    label="allskycam: index > 0.5 … > 0.2",
)
ax.plot(
    mo,
    clim.cam30 * 100,
    color=P["blue"],
    marker="o",
    ms=5,
    label="allskycam: fraction of dark time with cloud index > 0.3 (2021–26)",
)
ax.plot(
    mo,
    clim.CTIO_1975_2022 * 100,
    color=INK2,
    label="CTIO cloud DB 1975–2022, cloud > 3/8 (scheduler default)",
)
ax.plot(
    mo,
    clim.nowcast_proxy_2023_26 * 100,
    color=P["aqua"],
    label="meteoblue nowcast proxy 2023–26",
)
ax.plot(
    mo,
    clim.Scheduler_WEATHER_2026 * 100,
    lw=0,
    marker="s",
    ms=7,
    color=P["orange"],
    label="Scheduler WEATHER flag, Mar–Jul 2026",
)
ax.set_xticks(mo)
ax.set_xticklabels(MON)
ax.set_ylabel("% of dark time")
ax.set_ylim(0, 60)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
ax.set_title("Seasonal cloud climatology at Cerro Pachón")
plt.tight_layout()
clim.round(3)

In [ ]:
yr = N.groupby("year").agg(
    nights=("cloudy30", "size"),
    cam30=("cloudy30", "mean"),
    fully=("cloudy30", lambda s: (s > 0.9).mean()),
    clear=("cloudy30", lambda s: (s < 0.1).mean()),
)
eqm = lambda col: N.groupby("month")[col].mean().mean()
head = pd.DataFrame(
    {
        "allskycam, index>0.3": dict(
            annual_mean_equal_month=eqm("cloudy30"),
            nightly_median=N.cloudy30.median(),
            nightly_std=N.cloudy30.std(),
            nights_fully_lost=(N.cloudy30 > 0.9).mean(),
            nights_partial=((N.cloudy30 >= 0.1) & (N.cloudy30 <= 0.9)).mean(),
            nights_clear=(N.cloudy30 < 0.1).mean(),
            annual_std_across_years=yr.cam30.std(),
        ),
        "allskycam, index>0.5": dict(
            annual_mean_equal_month=eqm("cloudy50"),
            nightly_median=N.cloudy50.median(),
            nightly_std=N.cloudy50.std(),
            nights_fully_lost=(N.cloudy50 > 0.9).mean(),
            nights_partial=((N.cloudy50 >= 0.1) & (N.cloudy50 <= 0.9)).mean(),
            nights_clear=(N.cloudy50 < 0.1).mean(),
            annual_std_across_years=N.groupby("year").cloudy50.mean().std(),
        ),
        "allskycam, mean index": dict(
            annual_mean_equal_month=eqm("idx_mean"),
            nightly_median=N.idx_mean.median(),
            nightly_std=N.idx_mean.std(),
        ),
    }
).T
print("by year (index>0.3):")
print(yr.round(3).to_string())
head.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].hist(N.cloudy30, bins=np.linspace(0, 1, 21), color=P["blue"], zorder=2)
axes[0].set_xlabel("fraction of night with cloud index > 0.3")
axes[0].set_ylabel("nights")
axes[0].set_title("Nights are mostly clear or mostly lost")
yy = N.groupby(N.index // 100).cloudy30.mean()
x = pd.to_datetime(yy.index.astype(str), format="%Y%m")
axes[1].plot(x, yy * 100, color=P["blue"], lw=1.5)
axes[1].set_ylabel("% dark time cloudy (idx>0.3)")
axes[1].set_title(
    "Monthly series: 2023 was the cloudiest year, 2022 and 2026 the clearest"
)
plt.tight_layout()

### 5.1 Where the clouds are

Nightly mean cloud index per NSIDE-8 horizon pixel. Averaged over all nights the index is nearly flat with altitude (0.15 at the zenith, 0.17 at 10–60°) and with azimuth (0.14–0.18, with a weak southern maximum of 0.22). On nights the allskycam calls clear overall it is 0.02–0.06 in every direction except the south (0.10).

In v3 the same plot showed a strong north / north-west excess on clear nights (index 0.61 at N vs 0.04–0.17 elsewhere), and this section attributed it to the mast obstructing the horizon. That excess has gone. The per-pixel table behind it was built at 10:59 on 2026-10-04, before the v3 re-run finished (the completeness model was rebuilt at 14:38), and was not rebuilt afterwards (`build_index_tables.py` now does this in `finalize_rerun.sh`); the v1 mis-registered astrometry made the north/west half of the sky "cloudy" on 425 nights (§2.2), which is the likely source. It was a reduction artefact, not an obstruction. The residual southern excess is small, but it is present on clear nights too, so it should still be checked against the frames before any per-pixel use.

In [ ]:
import healpy as hp

PX = pd.read_parquet(A / "index/cloud_pix8_nightly.parquet")
mean_pix = PX.mean(axis=0).values
th, ph = hp.pix2ang(8, np.arange(768))
alt = 90 - np.degrees(th)
az = np.degrees(ph)
ok = np.isfinite(mean_pix) & (alt > 10)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ab = pd.cut(alt[ok], [10, 20, 30, 45, 60, 90])
s = pd.Series(mean_pix[ok]).groupby(ab, observed=True).mean()
axes[0].bar(range(len(s)), s.values, color=P["blue"], zorder=2)
axes[0].set_xticks(range(len(s)))
axes[0].set_xticklabels([str(i) for i in s.index], fontsize=8)
axes[0].set_xlabel("altitude band (deg)")
axes[0].set_ylabel("mean cloud index, all nights")
axes[0].set_title("Cloud index vs altitude")
azb = pd.cut(
    az[ok & (alt < 45)],
    np.arange(0, 361, 45),
    labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
)
s2 = pd.Series(mean_pix[ok & (alt < 45)]).groupby(azb, observed=True).mean()
clear_n = N[N.cloudy30 < 0.05].index
mc = PX.loc[PX.index.isin(clear_n)].mean(axis=0).values
s3 = pd.Series(mc[ok & (alt < 45)]).groupby(azb, observed=True).mean()
w = 0.4
axes[1].bar(
    np.arange(len(s2)) - w / 2,
    s2.values,
    w,
    color=P["blue"],
    zorder=2,
    label="all nights",
)
axes[1].bar(
    np.arange(len(s2)) + w / 2,
    s3.values,
    w,
    color=P["orange"],
    zorder=2,
    label="allskycam-clear nights only",
)
axes[1].set_xticks(range(len(s2)))
axes[1].set_xticklabels(s2.index)
axes[1].set_xlabel("azimuth (alt < 45°)")
axes[1].set_title("Azimuth dependence, all nights vs clear nights")
axes[1].legend(frameon=False, fontsize=8.5)
plt.tight_layout()

## 6. Transparency, sky brightness and streaks

**Transparency.** On clear frames the matched stars give a zero point (V = m_inst + zp − k·X). Absolute zp depends on the camera unit and its plate scale (six body/lens swaps 2021–26, see `build_unit_history.py`), so transparency is expressed *relative to the clear-sky median zp of the same month and unit*; since the v5 re-reduction every night uses the same peak-based flux. Nights 0.1 mag below the monthly median are "non-photometric" by this allskycam (its own night-to-night scatter is 0.04 mag).

In [ ]:
FR = pd.read_parquet(
    A / "index/frames_all_regated.parquet",
    columns=[
        "day_obs",
        "seq",
        "zp",
        "zp_scatter",
        "n_phot",
        "pipe_version",
        "unit",
        "night",
        "sun_alt",
        "moon_alt",
        "fd5_far",
        "sky_zenith_adu_s",
    ],
)
FR = FR[(FR.pipe_version >= 5) & (FR.sun_alt < -18) & (FR.n_phot >= 100)].merge(
    CL[["day_obs", "seq", "cloud_idx"]], on=["day_obs", "seq"]
)  # one flux definition across the archive: v5 introduced the peak-based flux, v6 changed only the astrometry
clear = FR[FR.cloud_idx < 0.1]
nz = clear.groupby(["night", "unit"]).zp.median()
mon = nz.groupby(
    [nz.index.get_level_values("night") // 100, nz.index.get_level_values("unit")]
).transform(
    "median"
)  # absolute zp differs between camera units and focal lengths
dz = (nz - mon).rename("dzp").droplevel("unit")
dz = dz.dropna()
print(
    f"{len(dz)} nights with a clear-sky zero point; night-to-night scatter about the monthly median: {1.4826*np.median(np.abs(dz)):.3f} mag; nights > 0.1 mag below: {(dz < -0.1).mean():.1%}; > 0.2 mag: {(dz < -0.2).mean():.1%}"
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
axes[0].hist(dz, bins=np.linspace(-0.6, 0.3, 46), color=P["blue"], zorder=2)
axes[0].set_xlabel("nightly clear-sky zp − monthly median (mag)")
axes[0].set_ylabel("nights")
axes[0].set_title("Transparency on allskycam-clear nights")
sk = N.sky_zenith_dark_med.dropna()
axes[1].scatter(
    pd.to_datetime(sk.index.astype(str), format="%Y%m%d"),
    sk,
    s=6,
    color=P["blue"],
    alpha=0.6,
    edgecolor="none",
)
axes[1].set_ylabel("zenith sky, moonless clear (ADU/s, green)")
axes[1].set_title(
    "Dark-sky brightness (uncalibrated; epoch steps are plate-scale changes)"
)
plt.tight_layout()

In [ ]:
# Streaks: interior (r/R<0.8, length>60 px) components on clear dark frames; rate per frame by year
import json

rows = []
fr_small = pd.read_parquet(
    A / "index/frames_all_regated.parquet",
    columns=["day_obs", "seq", "sun_alt", "disk_cx", "disk_cy", "disk_R"],
)
for f in sorted(glob.glob(str(A / "streaks/*.parquet")))[::2]:
    d = int(f.split("/")[-1][:8])
    s = pd.read_parquet(f)
    if len(s) == 0 or "x0" not in s:
        continue
    s["day_obs"] = d
    rows.append(s)
S = (
    pd.concat(rows)
    .merge(fr_small, on=["day_obs", "seq"], how="left")
    .merge(CL[["day_obs", "seq", "cloud_idx"]], on=["day_obs", "seq"], how="left")
)
S["r"] = (
    np.hypot((S.x0 + S.x1) / 2 - S.disk_cx, (S.y0 + S.y1) / 2 - S.disk_cy) / S.disk_R
)
inner = S[(S.r < 0.8) & (S.length_px > 60) & (S.cloud_idx < 0.1) & (S.sun_alt < -18)]
frames = CL[
    (CL.sun_alt < -18) & (CL.cloud_idx < 0.1) & CL.day_obs.isin(S.day_obs.unique())
]
rate = (
    inner.groupby(inner.day_obs // 10000).size()
    / frames.groupby(frames.day_obs // 10000).size()
).rename("interior streaks per clear dark frame")
tw = S[(S.r < 0.8) & (S.length_px > 60) & (S.cloud_idx < 0.1) & (S.sun_alt > -18)]
ftw = CL[
    (CL.sun_alt > -18)
    & (CL.sun_alt < -12)
    & (CL.cloud_idx < 0.1)
    & CL.day_obs.isin(S.day_obs.unique())
]
print(
    f"edge components (r/R>0.8: horizon structures, dome, mast): {(S.r > 0.8).mean():.0%} of all 'streaks' — excluded"
)
print(
    f"interior streak length: median {inner.length_px.median()*4.5/60:.0f}°, 90th pct {inner.length_px.quantile(.9)*4.5/60:.0f}° per 30 s frame (4.5′/px);  twilight rate {len(tw)/len(ftw):.2f} vs dark {len(inner)/len(frames):.2f} per frame"
)
pd.DataFrame(rate).round(2).T

Streak rates are a by-product and not yet validated against a satellite catalogue; the twilight excess (sunlit low-Earth-orbit satellites) and the sub-degree-per-second lengths are the right qualitative signatures. Treat the per-year trend as indicative only, since the detection threshold scales with the per-month noise.

### 6.1 Deep index: thin cloud and the empty half of the map

The standard index counts BSC stars with V < 5.5. On a clear dark frame that leaves only about 105 of the 272 NSIDE-8 cells above 20° with the 3 expected stars a cell needs, so the per-cell maps (§5.1) are undefined over 60 % of the sky. Those stars are also 1.5–2 mag brighter than the detection limit, so thin cloud barely changes their count: 0.5 mag of grey extinction removes about 1 % of them. The index sees thick cloud, not thin cloud.

After the 2026-10-16 refocus (star FWHM 1.4 px) the camera detects stars to V ≈ 7. A **deep index** now runs alongside the standard one. It uses the same detections and per-frame astrometry, with no CR2 re-read (`deep_match.py` → `build_completeness_deep.py` → `cloud_metric_deep.py`, see [rubin_allskycam/README.md](https://github.com/bstalder/rubin_allskycam#readme)):

* catalog: Hipparcos V < 7.5 (19,700 stars), with proper motion applied and stars dropped that have a brighter neighbour within 0.3°. The match radius is 2 px.
* clear-sky model: per service stint × month as for the standard index, with finer V bins above V 6. Reference frames are ranked on the deep stars' own completeness, so thin-cloud frames are not used as "clear".
* `cloud_idx_deep` = 1 − observed/expected, and **`ext_deep`** = the grey extinction (mag, 0–3) that brings the model's expected count down to the observed one. Cell maps are at NSIDE 8 and NSIDE 16.

The standard index stays the five-year record. The deep stars are only well detected on the sharp stints, so the deep index's sensitivity changes with focus.

In [ ]:
DF = pd.read_parquet(A / "index/deep_cloud_all_frames.parquet")
TD = pd.read_parquet(A / "calib/clear_sky_completeness_deep.parquet")
dk = DF[DF.sun_alt < -18].copy()
dk["stint"] = dk.model_key.str.split("|").str[-1]
c0 = dk[dk.cloud_idx == 0]  # frames the standard index calls fully clear
v7 = TD[(TD.alt_lo >= 45) & (TD.sky_lo < 1.2) & (TD.Vlo == 7.0)].groupby("key")
deep_tab = pd.DataFrame(
    dict(
        dark_frames=dk.groupby("stint").size(),
        p_det_V7_clear=v7.n_det.sum() / v7.n.sum(),
        cells8_standard=c0.groupby("stint").n_cells.median(),
        cells8_deep=c0.groupby("stint").n_cells8.median(),
        cells16_deep=c0.groupby("stint").n_cells16.median(),
        clear_floor_idx=c0.groupby("stint").cloud_idx_deep.median(),
        clear_floor_ext=c0.groupby("stint").ext_deep.median(),
        thin_frac=dk.assign(t=(dk.cloud_idx < 0.1) & (dk.ext_deep > 0.2))
        .groupby("stint")
        .t.mean(),
    )
).dropna(subset=["dark_frames"])
# DREAM whole-sky extinction at each allskycam frame (nearest map within 60 s), 2025-06 -> 2026-06
dm = pd.read_parquet(
    SP / "dream_maps.parquet", columns=["mjd", "med_ext_alt30"]
).sort_values("mjd")
dd = pd.merge_asof(
    dk[["mjd", "cloud_idx", "cloud_idx_deep", "ext_deep"]]
    .dropna(subset=["mjd"])
    .sort_values("mjd"),
    dm,
    on="mjd",
    tolerance=60 / 86400,
    direction="nearest",
).dropna(subset=["med_ext_alt30", "ext_deep"])
thin = dd[dd.cloud_idx < 0.05]
EB = [0, 0.05, 0.1, 0.2, 0.4, 0.8, 3.0]
tb = thin.groupby(
    pd.cut(thin.ext_deep, EB, include_lowest=True), observed=True
).med_ext_alt30
tb = pd.DataFrame(
    dict(
        frames=tb.size(),
        dream_median=tb.median(),
        dream_q25=tb.quantile(0.25),
        dream_q75=tb.quantile(0.75),
    )
)
print(
    f"{len(dk):,} dark frames; on clear frames the deep index covers {c0.n_cells8.median():.0f} of 272 NSIDE-8 cells "
    f"(standard {c0.n_cells.median():.0f}); {((dk.cloud_idx < 0.1) & (dk.ext_deep > 0.2)).mean():.1%} of dark frames are "
    f"clear by the standard index but have ext_deep > 0.2 mag"
)
print(
    f"DREAM overlap: {len(dd):,} frames; frames with standard index < 0.05, DREAM median extinction by ext_deep bin:"
)
display(tb.round(3))
display(deep_tab.round(3))

fig, axes = plt.subplots(
    1, 2, figsize=(10.5, 3.9), gridspec_kw=dict(width_ratios=[1, 1.1])
)
cov = deep_tab[["cells8_standard", "cells8_deep"]].iloc[::-1] / 272
y = np.arange(len(cov))
h = 0.4
axes[0].barh(
    y + h / 2,
    cov.cells8_standard,
    h * 0.9,
    color=P["blue"],
    zorder=2,
    label="standard (BSC V<5.5)",
)
axes[0].barh(
    y - h / 2,
    cov.cells8_deep,
    h * 0.9,
    color=P["orange"],
    zorder=2,
    label="deep (Hipparcos V<7.5)",
)
axes[0].set_yticks(y)
axes[0].set_yticklabels(
    [
        f"{s.split('@')[0]}  from {pd.Timestamp(s.split('@')[1]):%Y-%m-%d}"
        for s in cov.index
    ],
    fontsize=8,
)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("fraction of alt>20° NSIDE-8 cells with ≥ 3 expected stars")
axes[0].set_title("Sky coverage on clear dark frames, by stint")
axes[0].legend(
    frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2
)
axes[0].grid(axis="y", visible=False)
xc = np.arange(len(tb))
axes[1].fill_between(
    xc,
    tb.dream_q25,
    tb.dream_q75,
    color=P["orange"],
    alpha=0.2,
    lw=0,
    zorder=2,
    label="DREAM interquartile range",
)
axes[1].plot(
    xc,
    tb.dream_median,
    color=P["orange"],
    marker="o",
    ms=8,
    zorder=3,
    label="DREAM median",
)
axes[1].set_xticks(xc)
axes[1].set_xticklabels([f"{a:g}–{b:g}" for a, b in zip(EB[:-1], EB[1:])], fontsize=8)
for xi, (n, v) in enumerate(zip(tb.frames, tb.dream_median)):
    axes[1].annotate(
        f"{n:,}",
        (xi, v),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        fontsize=7.5,
        color=INK2,
    )
axes[1].set_xlabel("allskycam ext_deep (mag); labels: frames per bin")
axes[1].set_ylabel("DREAM extinction, alt>30° (mag)")
axes[1].set_title("Frames the standard index calls clear: DREAM vs ext_deep")
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** On clear dark frames the deep index covers about 240 of the 272 cells, against about 105 for the standard one, in every stint. At NSIDE 16 (3.7° cells) it covers 460–725 cells on the sharp stints (8323/0552 since 2026-10-16, 9118/0165 in 2022, 9127/0545 in 2023). Its clear-sky floor is a median index of 0.01–0.02 (≈ 0.02 mag) in every stint except the six-night 8323/0552@20261010 stint (0.09), which never had enough clear frames for its own reference.

Thin cloud: about 6 % of dark frames are clear by the standard index but have `ext_deep` > 0.2 mag. DREAM agrees on these frames. Where `ext_deep` is below 0.2 mag the DREAM median is 0.04 mag, at 0.2–0.4 mag it is 0.07 mag, and at 0.4–0.8 mag it is 0.28 mag. In October 2026, 11–38 % of the dark frames on 10-17, 10-22 and 10-28 were thin cloud of this kind, all nights the standard index calls clear.

Limits: completeness at V 7.0–7.25 on a clear dark sky is 0.76–0.84 on the two sharpest stints (8323/0552 since 2026-10-16, 9118/0165 in 2022), 0.51 for 9127/0545, and 0.02–0.23 on the others (FWHM 2.6–3.5 px). On those stints the deep index is close to a V < 6 index, and `ext_deep` saturates near 1 mag. Frame by frame it does not track DREAM better than the standard index for thick cloud (rank correlation with DREAM extinction 0.42 vs 0.43). Its gain is coverage and thin cloud. Per-frame values are in `index/deep_cloud_all_frames.parquet`, nightly values in `index/deep_nightly.parquet`.

In [ ]:
out = dict(
    nights=int(len(N)),
    frames=int(len(CL)),
    flag_validation=roc.to_dict(),
    monthly_climatology=clim.round(4).to_dict(),
    headline=head.round(4).to_dict(orient="index"),
    by_year=yr.round(4).to_dict(orient="index"),
    dream=dream_res,
)
files = [
    "allsky_cloud_results.json",
    "allsky_monthly_cloud_climatology.csv",
    "allsky_nightly_summary.parquet",
    "dream_nightly.parquet",
]
if WRITE_RESULTS:
    json.dump(out, open(SP / files[0], "w"), indent=1, default=str)
    clim.to_csv(SP / files[1])
    N.to_parquet(SP / files[2])
    dv.to_parquet(SP / files[3])
    print(f"saved to {SP}: " + ", ".join(files))
else:
    print(f"not saved (set ROSP11_WRITE_RESULTS=1 to write {', '.join(files)} to {SP})")